In [3]:
r"""
Скрипт за създаване на осреднени композитни изображения от месечните
10×10 км квадрати (Sentinel‑2 L2A) за 2024 г.
За всеки пожар се изчислява средноаритметично на каналите от наличните
месечни изображения (май, юли, септември), като нулевите пиксели (nodata)
се игнорират. Изображенията с различни размери се преоразмеряват към
обща референтна мрежа.

Script for creating averaged composite images from the monthly
10×10 km squares (Sentinel‑2 L2A) for 2024.
For each fire, the arithmetic mean of the bands is computed from the
available monthly images (May, July, September), ignoring zero (nodata) pixels.
Images with differing sizes are resampled to a common reference grid.

Вход / Input:   D:\data\master_thesis\exports\sentinel2_fire_images\composite_pictures
Изход / Output: D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures
"""

import os
import re
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from collections import defaultdict

# ======================== ПЪТИЩА / PATHS ============================
INPUT_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\composite_pictures'
OUTPUT_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures'

# ======================== КОНФИГУРАЦИЯ / CONFIGURATION ====================
NODATA_VALUE = 0          # стойност, която се игнорира при осредняване / value ignored during averaging
YEAR = 2024               # целева година / target year
MONTHS = ['may', 'july', 'september']   # целеви месеци / target months

def parse_fire_id_from_filename(filename):
    """
    Извлича ID на пожара от името на файла.
    Extract fire ID from filename.
    Очакван формат: square_10km_allbands_2024_<month>_<fire_id>_<date>.tif
    Expected format: square_10km_allbands_2024_<month>_<fire_id>_<date>.tif
    """
    # Example: square_10km_allbands_2024_may_1_20240512.tif
    pattern = r'square_10km_allbands_(\d{4})_([a-z]+)_(\d+)_(\d{8})\.tif'
    match = re.match(pattern, filename)
    if match:
        year = int(match.group(1))
        month = match.group(2)
        fire_id = match.group(3)
        return year, month, fire_id
    return None, None, None

def main():
    print("Създаване на осреднени 10×10 км композити за 2024 г.")
    print("Creating averaged 10×10 km composites for 2024")
    print("=" * 70)
    print(f"Входна директория / Input dir: {INPUT_DIR}")
    print(f"Изходна директория / Output dir: {OUTPUT_DIR}")
    print(f"Nodata стойност / Nodata value: {NODATA_VALUE}")
    print("=" * 70)

    # Създаване на изходна папка / Create output folder
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Събиране на файловете по пожар / Group files by fire
    fire_groups = defaultdict(list)

    for fname in os.listdir(INPUT_DIR):
        if fname.lower().endswith('.tif'):
            year, month, fire_id = parse_fire_id_from_filename(fname)
            if year == YEAR and month in MONTHS and fire_id is not None:
                fire_groups[fire_id].append(os.path.join(INPUT_DIR, fname))

    if not fire_groups:
        print("❌ Не са намерени подходящи файлове / No suitable files found.")
        return

    print(f"\nНамерени {len(fire_groups)} пожара с файлове.\n")

    # Обработка на всеки пожар / Process each fire
    for fire_id, file_list in sorted(fire_groups.items(), key=lambda x: int(x[0])):
        print(f"--- Пожар / Fire {fire_id} ---")
        # Сортиране по месец / Sort by month (optional)
        file_list.sort()
        print(f"  Файлове / Files: {[os.path.basename(f) for f in file_list]}")

        # Определяне на референтна мрежа от първия файл / Define reference grid from first file
        with rasterio.open(file_list[0]) as ref_src:
            ref_crs = ref_src.crs
            ref_transform = ref_src.transform
            ref_height = ref_src.height
            ref_width = ref_src.width
            band_count = ref_src.count
            band_descriptions = [ref_src.descriptions[i] for i in range(band_count)]
            ref_dtype = ref_src.dtypes[0]  # assume all bands same dtype

        print(f"  Референтна мрежа / Reference grid: {ref_width}×{ref_height}, CRS={ref_crs.to_epsg()}, dtype={ref_dtype}")

        # Проверка на CRS съвместимост / Check CRS compatibility
        all_crs_ok = True
        for fpath in file_list[1:]:
            with rasterio.open(fpath) as src:
                if src.crs != ref_crs:
                    print(f"  ⚠ Различен CRS в {os.path.basename(fpath)}: {src.crs} != {ref_crs}")
                    all_crs_ok = False
        if not all_crs_ok:
            print("  ❌ Пропускане поради различен CRS / Skipping due to CRS mismatch.")
            continue

        # Инициализиране на суматор и брояч / Initialize sum and count arrays
        sum_bands = np.zeros((band_count, ref_height, ref_width), dtype=np.float64)
        count_bands = np.zeros((band_count, ref_height, ref_width), dtype=np.uint16)

        # Обработка на всяко изображение / Process each image
        for idx, fpath in enumerate(file_list):
            print(f"  📥 Обработка / Processing: {os.path.basename(fpath)}")
            with rasterio.open(fpath) as src:
                # If same grid, read directly; otherwise resample
                if (src.width == ref_width and src.height == ref_height and
                    src.transform == ref_transform):
                    data = src.read()
                    print("     ✓ Същата мрежа, директно четене / Same grid, direct read")
                else:
                    print(f"     ⚠ Различен размер: {src.width}×{src.height}, преоразмеряване / resampling")
                    # Reproject to reference grid
                    data = np.zeros((band_count, ref_height, ref_width), dtype=ref_dtype)
                    reproject(
                        source=src.read(),
                        destination=data,
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=ref_transform,
                        dst_crs=ref_crs,
                        resampling=Resampling.bilinear
                    )

            # Натрупване на валидни пиксели / Accumulate valid pixels
            valid_mask = data > NODATA_VALUE
            sum_bands += data.astype(np.float64) * valid_mask.astype(np.float64)
            count_bands += valid_mask.astype(np.uint16)

        # Изчисляване на средното / Compute mean
        with np.errstate(invalid='ignore'):
            mean_bands = np.where(count_bands > 0,
                                  sum_bands / count_bands,
                                  NODATA_VALUE)
        # Възстановяване на оригиналния тип данни / Restore original dtype
        mean_bands = mean_bands.astype(ref_dtype)

        # Запис на изходния файл / Write output file
        out_filename = f"fire{fire_id}_simple_average_all_bands_{YEAR}.tif"
        out_path = os.path.join(OUTPUT_DIR, out_filename)

        profile = {
            'driver': 'GTiff',
            'height': ref_height,
            'width': ref_width,
            'count': band_count,
            'dtype': mean_bands.dtype,
            'crs': ref_crs,
            'transform': ref_transform,
            'compress': 'deflate',
            'nodata': NODATA_VALUE
        }

        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(mean_bands)
            for i, desc in enumerate(band_descriptions, 1):
                dst.set_band_description(i, desc)

            # Добавяне на метаданни / Add metadata
            tags = {
                'fire_id': fire_id,
                'year': str(YEAR),
                'months': ', '.join(MONTHS),
                'composite_type': 'simple_average',
                'nodata_ignored': str(NODATA_VALUE),
                'source_files': ', '.join([os.path.basename(f) for f in file_list]),
                'number_of_inputs': str(len(file_list))
            }
            dst.update_tags(**tags)

        print(f"  ✅ Записан / Saved: {out_filename}")
        print(f"     Размер / Size: {ref_width}×{ref_height}, канали / bands: {band_count}")
        print()

    print("=" * 70)
    print("ОБРАБОТКАТА ЗАВЪРШИ / PROCESSING COMPLETED.")
    created_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith('fire') and f.endswith('.tif')]
    print(f"Създадени композитни файлове / Composite files created: {len(created_files)}")

if __name__ == "__main__":
    main()

Създаване на осреднени 10×10 км композити за 2024 г.
Creating averaged 10×10 km composites for 2024
Входна директория / Input dir: D:\data\master_thesis\exports\sentinel2_fire_images\composite_pictures
Изходна директория / Output dir: D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures
Nodata стойност / Nodata value: 0

Намерени 16 пожара с файлове.

--- Пожар / Fire 1 ---
  Файлове / Files: ['square_10km_allbands_2024_july_1_20240708.tif', 'square_10km_allbands_2024_may_1_20240623.tif', 'square_10km_allbands_2024_september_1_20240926.tif']
  Референтна мрежа / Reference grid: 1014×1010, CRS=32635, dtype=uint16
  📥 Обработка / Processing: square_10km_allbands_2024_july_1_20240708.tif
     ✓ Същата мрежа, директно четене / Same grid, direct read
  📥 Обработка / Processing: square_10km_allbands_2024_may_1_20240623.tif
     ✓ Същата мрежа, директно четене / Same grid, direct read
  📥 Обработка / Processing: square_10km_allbands_2024_september_1_20240926.tif
     ✓ 